# 滑动窗口记忆

> **只保留最后 *k* 轮对话：在保持对话连贯性的同时限制记忆成本的最直接方式。**

在[上一份 notebook](../01_conversation_buffer_memory/conversation_buffer_memory.ipynb)中我们看到**对话缓冲记忆**会存储所有内容。这导致每轮输入 token 线性增长，累积成本二次增长。对于大多数聊天应用来说，这是很浪费的——用户很少引用 30 轮之前说过的话。

想象一块只能贴五张便利贴的白板。当你贴第六张时，你会撕掉最旧的那张扔掉。**滑动窗口记忆**以同样的方式工作：它保留一个固定大小的最近消息窗口，丢弃所有更旧的内容。最后 *k* 条消息始终可用，但更早的消息会消失。

**完成本 notebook 后你将理解：**
- 如何使用 LangChain 从头构建基于 deque 的滑动窗口。
- 窗口大小、回忆能力和 token 成本之间的精确权衡。
- 如何运行一个小型评估来衡量不同窗口大小下的回忆表现。
- 滑动窗口记忆何时是正确的（以及错误的）选择。

## 核心概念

- **窗口大小 *k***：保留的消息最大数量。更大的 *k* 意味着更好的回忆能力，但 token 成本更高。
- **FIFO 驱逐**：FIFO 代表"先进先出"。当 `len(messages) > k` 时，最旧的消息从前端被丢弃，窗口向前"滑动"。
- **`collections.deque`**：Python 的双端队列，带有 `maxlen` 参数。天然适合：向已满的 deque 追加元素会自动丢弃最旧的项。
- **近因偏差**：按照设计，Agent 记住最近的上下文而遗忘更早的上下文。当最近上下文才是最重要的内容时，这是一个特性而非缺陷。
- **消息计数 vs. 轮次计数窗口**：*k* 条消息的窗口计算每条独立消息（用户或助手）。*k* 轮对话的窗口保留最后 *k* 对用户-助手组合。轮次计数窗口避免产生孤立消息（没有对应问题的回复）。
- **恒定的 token 成本**：与缓冲记忆不同，每轮 token 成本受窗口大小限制。这为你提供可预测的延迟和成本。

## 模型准备

导入 LangChain 和标准辅助工具。我们还从 Python 的 `collections` 模块引入 `deque`。**deque**（读作"deck"）是一个双端队列，可以容纳固定数量的项。当你再添加一项时，最旧的项会自动被丢弃。

In [1]:
# 导入Langchain的初始化模型的函数
from langchain.chat_models import init_chat_model
# 加载环境变量
from dotenv import load_dotenv
load_dotenv()

# 调用init_chat_model函数初始化模型，参数model用来指定模型名称，Langchain会根据模型名字自动设定base_url，并从环境变量中获取api_key
model = init_chat_model(model="deepseek-chat")
print(type(model)) # <class 'langchain_deepseek.chat_models.ChatDeepSeek'>

/Users/huanglu/Project/ai-agent-notes/.venv/lib/python3.11/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


<class 'langchain_deepseek.chat_models.ChatDeepSeek'>


## 核心实现

关键洞察：Python 的 `collections.deque(maxlen=k)` 免费提供滑动窗口行为。当你对一个已满的 deque 执行 `append()` 时，另一端的最旧项会自动被丢弃。

In [ ]:
import copy
from collections import deque
from langchain_core.messages import HumanMessage, AIMessage, SystemMessage

class SlidingWindowMemory:
    """滑动窗口记忆，只保留最后 *k* 条消息。"""

    def __init__(
        self,
        window_size: int = 10,
        model_name: str = "deepseek-chat",
        system_prompt: str | None = None,
        max_tokens: int = 1024,
    ):
        self.llm = init_chat_model(model=model_name)
        self.system_prompt = system_prompt
        self.max_tokens = max_tokens

        # 滑动窗口 - 带有固定最大长度的双端队列
        self.window_size = window_size
        self.messages: deque[dict] = deque(maxlen=window_size)

        # 记录所有消息用于分析（不会发送给 LLM）
        self.full_history: list[dict] = []
        self.turn_token_usage: list[dict] = []

    def _to_langchain_messages(self) -> list:
        """将内部 deque 转为 LangChain 消息对象。"""
        lc_messages = []
        if self.system_prompt:
            lc_messages.append(SystemMessage(content=self.system_prompt))
        for msg in self.messages:
            if msg["role"] == "user":
                lc_messages.append(HumanMessage(content=msg["content"]))
            elif msg["role"] == "assistant":
                lc_messages.append(AIMessage(content=msg["content"]))
        return lc_messages

    # ── 聊天 ─────────────────────────────────────────────────────
    def chat(self, user_input: str) -> str:
        """发送一条消息；只有最后 *k* 条消息会被发送给 LLM。"""
        user_msg = {"role": "user", "content": user_input}
        self.messages.append(user_msg)
        self.full_history.append(user_msg)

        lc_messages = self._to_langchain_messages()
        response = self.llm.invoke(lc_messages)
        assistant_text = response.content

        assistant_msg = {"role": "assistant", "content": assistant_text}
        self.messages.append(assistant_msg)
        self.full_history.append(assistant_msg)

        # 记录本轮 token 使用量
        usage = response.usage_metadata or {}
        self.turn_token_usage.append({
            "turn": len(self.turn_token_usage) + 1,
            "input_tokens": usage.get("input_tokens", 0),
            "output_tokens": usage.get("output_tokens", 0),
        })
        return assistant_text

    # ── 检查辅助方法 ───────────────────────────────────────────
    def get_window(self) -> list[dict]:
        """返回当前窗口内容（即 LLM 看到的内容）。"""
        return list(self.messages)

    def get_full_history(self) -> list[dict]:
        """返回所有曾经发送过的消息，包括已被驱逐的。"""
        return copy.deepcopy(self.full_history)

    @property
    def evicted_count(self) -> int:
        """已被从窗口中移除的消息数量。"""
        return len(self.full_history) - len(self.messages)

    def clear(self) -> None:
        self.messages.clear()
        self.full_history.clear()
        self.turn_token_usage.clear()

    def __repr__(self) -> str:
        return (
            f"SlidingWindowMemory(window={len(self.messages)}/{self.window_size}, "
            f"total={len(self.full_history)} messages, "
            f"evicted={self.evicted_count})"
        )

print("✓ SlidingWindowMemory 类已定义")

## 使用示例：观察窗口滑动

我们使用一个小窗口（`k=6` 条消息，即 3 轮对话）来观察对话增长时发生的情况。我们会在早期植入一个事实，然后在它滑出窗口后询问它。

In [15]:
mem = SlidingWindowMemory(
    window_size=6,  # 3 个完整轮次（每轮包含用户 + 助手各一条）
    system_prompt="你是一个简洁的助手。用 1-2 句话回复。",
)

conversation = [
    "我叫 Alice，是一名飞行员。",           # 第 1 轮 - 植入一个事实
    "我在一家地区航空公司飞波音 737。",    # 第 2 轮 - 植入另一个事实
    "今天西雅图的天气怎么样？",            # 第 3 轮 - 无关的填充对话
    "我叫什么名字，做什么工作？",          # 第 4 轮 - 回忆测试（第 1 轮可能已被驱逐）
]

for msg in conversation:
    print(f"👤 用户:  {msg}")
    reply = mem.chat(msg)
    print(f"🤖 Agent: {reply}")
    print(f"   📊 窗口: {len(mem.messages)}/{mem.window_size} 条消息 | 已驱逐: {mem.evicted_count}")
    print()

👤 用户:  我叫 Alice，是一名飞行员。
🤖 Agent: 你好 Alice，飞行员是个令人尊敬的职业。需要我为你提供什么帮助吗？
   📊 窗口: 2/6 条消息 | 已驱逐: 0

👤 用户:  我在一家地区航空公司飞波音 737。
🤖 Agent: 驾驶波音737非常酷，这是一款经典机型。你对目前执飞的航线有什么特别的感受吗？
   📊 窗口: 4/6 条消息 | 已驱逐: 0

👤 用户:  今天西雅图的天气怎么样？
🤖 Agent: 抱歉，我无法实时获取天气信息。建议你开启联网搜索功能或查看专业气象应用获取最新数据。
   📊 窗口: 6/6 条消息 | 已驱逐: 0

👤 用户:  我叫什么名字，做什么工作？
🤖 Agent: 你是 Alice，在一家地区航空公司驾驶波音 737 的飞行员。
   📊 窗口: 6/6 条消息 | 已驱逐: 2



让我们检查 LLM 当前能看到的和已遗忘的内容。窗口内的消息对模型可见。被驱逐的消息则永远消失了。下面的例子把窗口缩小到 `window_size=4`，可以清楚看到第一对对话被遗忘了。

In [ ]:
mem = SlidingWindowMemory(
    window_size=4,  # 2 个完整轮次（每轮包含用户 + 助手各一条）
    system_prompt="你是一个简洁的助手。用 1-2 句话回复。",
)

for msg in conversation:
    print(f"👤 用户:  {msg}")
    reply = mem.chat(msg)
    print(f"🤖 Agent: {reply}")
    print(f"   📊 窗口: {len(mem.messages)}/{mem.window_size} 条消息 | 已驱逐: {mem.evicted_count}")
    print()

# 检查 LLM 当前看到的 vs. 它已遗忘的内容
print("=== 窗口内的消息（LLM 可以看到）===")
for i, msg in enumerate(mem.get_window()):
    role = "用户" if msg["role"] == "user" else "助手"
    preview = msg["content"][:70] + ("..." if len(msg["content"]) > 70 else "")
    print(f"  [{i}] {role}: {preview}")

print(f"\n=== 已驱逐的消息（LLM 无法看到）: {mem.evicted_count} ===")
evicted = mem.get_full_history()[:mem.evicted_count]
for i, msg in enumerate(evicted):
    role = "用户" if msg["role"] == "user" else "助手"
    preview = msg["content"][:70] + ("..." if len(msg["content"]) > 70 else "")
    print(f"  [{i}] {role}: {preview}")

## 讨论与权衡

### 优势
- **可预测的成本**：每轮 token 使用量受窗口大小限制，与对话长度无关。
- **快速构建**：Python 的 `deque(maxlen=k)` 一行代码就提供了整个机制。
- **对大多数聊天足够好**：用户很少引用 20+ 轮之前的内容。
- **低延迟**：有界的输入大小意味着有界的响应时间。

### 劣势
- **硬性信息截断**：一旦消息滑出窗口，它就消失了。没有痕迹，没有摘要，什么都没有。
- **没有优雅降级**：与摘要记忆不同，Agent 甚至不知道它遗忘了什么。
- **孤立消息**：消息计数窗口可能会拆分用户-助手配对，留下没有对应问题的回复。使用基于轮次的窗口可以避免这一点。
- **选择 *k* 很微妙**：太小会遗忘重要上下文，太大则失去了本技术的意义。

### 选择合适的窗口大小

| 窗口大小 | 适合场景 | 注意事项 |
|-------------|----------|-----------|
| k = 4-6 条消息 | 快速问答、无状态任务 | 2-3 轮后遗忘上下文 |
| k = 10-20 条消息 | 大多数聊天机器人、客服 | 成本上升；可能仍遗忘早期事实 |
| k = 40+ 条消息 | 复杂的多步骤任务 | 接近缓冲记忆成本；考虑混合方案 |

### 经验法则
从 `k = 10`（5 个完整轮次）开始，根据你的回忆需求和预算进行调整。

### 何时使用其他方案
- 如果需要记住更早的事实：使用**摘要记忆**或**实体记忆**。
- 如果需要平衡近因性和成本但无法承受硬截断：使用**摘要 + 缓冲混合方案**。
- 如果需要保持在精确的 token 预算内：使用 **Token 缓冲记忆**。

## 进一步阅读

- [Anthropic Messages API: 多轮对话](https://docs.anthropic.com/en/docs/build-with-claude/conversational-ai?utm_source=nirdiamant&utm_medium=github&utm_campaign=agent_memory_techniques)
- [LangChain ConversationBufferWindowMemory](https://python.langchain.com/docs/modules/memory/types/buffer_window?utm_source=nirdiamant&utm_medium=github&utm_campaign=agent_memory_techniques)
- [LlamaIndex ChatMemoryBuffer](https://docs.llamaindex.ai/en/stable/api_reference/memory/chat_memory_buffer/?utm_source=nirdiamant&utm_medium=github&utm_campaign=agent_memory_techniques)
- [Lilian Weng, "LLM Powered Autonomous Agents"（记忆章节）](https://lilianweng.github.io/posts/2023-06-23-agent/?utm_source=nirdiamant&utm_medium=github&utm_campaign=agent_memory_techniques)
- Python `collections.deque`：[官方文档](https://docs.python.org/3/library/collections.html#collections.deque)

---

*← 上一章：01 对话缓冲记忆 · 下一章：[03 摘要记忆](../03_summary_memory/) →*

![](https://europe-west1-amt-views-tracker.cloudfunctions.net/amt-tracker?notebook=all-techniques--02-sliding-window-memory--sliding-window-memory)
